# Capítulo 10 · Delineamentos experimentais e quase-experimentais em saúde: caderno complementar
*Ciência de Dados e Inteligência Artificial em Saúde, Volume 1*

> **Este caderno acompanha o capítulo do livro; não o reproduz.** Definições, deduções e explicações estão no livro. Aqui você executa os cálculos do caso e confere os números. Todos os dados são **sintéticos** e os valores têm apenas finalidade didática; não possuem correlação com o mundo real.

A **Parte A** percorre o caso do capítulo, o registro de um serviço de hipertensão com dez ambulatórios, pelas sete perguntas do diretor. A **Parte B** traz o código e o resultado de cada exercício do capítulo que exige código (os enunciados estão no livro). Execute as células em ordem. As saídas do código usam ponto decimal.

## Parte A. O caso

**Dados.** `data/clinic_patients.csv`: 6.744 linhas, uma por adulto que iniciou o tratamento ao longo de 24 meses, e dez colunas: `patient_id`, `clinic`, `month`, `period`, `clinic_start_month`, `diabetes`, `baseline_sbp`, `allocated`, `received`, `uncontrolled_12w` (sim/não escritos `yes`/`no`). Cada célula leva o número da seção do livro (10.1, 10.2, ...), para você acompanhar o capítulo e o caderno lado a lado.

### 10.1 Introdução: os dados

In [1]:
# 1. Ler o registro: cada linha é um adulto
import pandas as pd
from math import sqrt, log, exp
from statistics import NormalDist
from scipy.stats import t

df = pd.read_csv("data/clinic_patients.csv")

# 2. Contar linhas e colunas
rows, columns = df.shape
print("O registro tem", rows, "linhas (uma por adulto) e", columns, "colunas.")

# 3. Uma coluna Verdadeiro/Falso que é True quando a pressão continua não controlada em 12 semanas (usada daqui em diante)
df["y"] = df["uncontrolled_12w"] == "yes"
df.head()                                      # As cinco primeiras linhas

O registro tem 6744 linhas (uma por adulto) e 10 colunas.


,patient_id,clinic,month,period,clinic_start_month,diabetes,baseline_sbp,allocated,received,uncontrolled_12w,y
0,A00001,C01,1,before,19,no,149,separate tablets,no,no,False
1,A00002,C01,1,before,19,no,144,separate tablets,no,no,False
2,A00003,C01,1,before,19,no,159,separate tablets,no,no,False
3,A00004,C01,1,before,19,no,150,separate tablets,no,yes,True
4,A00005,C01,1,before,19,no,155,separate tablets,no,yes,True


### 10.2 A comparação e o risco do desfecho

*O que exatamente é comparado e qual é o risco de pressão não controlada no ensaio?*

In [2]:
# 1. Os meses do ensaio são as linhas com period "trial" (meses 7-12)
trial = df[df["period"] == "trial"]

# 2. Contar os adultos e os com pressão não controlada em 12 semanas
n_trial = len(trial)
events = trial["y"].sum()

# 3. Dividir a contagem por todos os adultos do ensaio
print("Adultos nos meses do ensaio:", n_trial)
print("Não controlados em 12 semanas:", events, "de", n_trial, "=", round(100 * events / n_trial, 1), "%")
print("Risco geral:", round(events / n_trial, 4))

Adultos nos meses do ensaio: 1716
Não controlados em 12 semanas: 478 de 1716 = 27.9 %
Risco geral: 0.2786


### 10.3.1 Randomização e equilíbrio basal

*Como tornar os dois grupos comparáveis e mantê-los assim?*

In [3]:
# 1. Pegar cada braço alocado e contar seus adultos
for arm in ["single pill", "separate tablets"]:
    group = trial[trial["allocated"] == arm]
    n = len(group)

    # 2. Contar os adultos com pressão sistólica basal de pelo menos 160 mmHg
    high = (group["baseline_sbp"] >= 160).sum()

    # 3. Contar os adultos com diabetes
    diabetes = (group["diabetes"] == "yes").sum()
    print(arm + ":", n, "adultos;", high, "com pressão basal de pelo menos 160 mmHg =", round(100 * high / n, 1), "%;",
          diabetes, "com diabetes =", round(100 * diabetes / n, 1), "%")

single pill: 857 adultos; 341 com pressão basal de pelo menos 160 mmHg = 39.8 %; 223 com diabetes = 26.0 %
separate tablets: 859 adultos; 331 com pressão basal de pelo menos 160 mmHg = 38.5 %; 207 com diabetes = 24.1 %


### 10.4.2 Intenção de tratar e incerteza

*Como o ensaio é analisado e o que significa sua diferença?*

In [4]:
# 1. Em cada braço alocado, contar os adultos e os não controlados
single = trial[trial["allocated"] == "single pill"]
separate = trial[trial["allocated"] == "separate tablets"]
a, n = single["y"].sum(), len(single)
b, m = separate["y"].sum(), len(separate)
print("Pílula única:", a, "de", n)
print("Comprimidos separados:", b, "de", m)

# 2. Risco = adultos não controlados / todos os adultos do braço
p = a / n
q = b / m

# 3. Diferença de riscos (pontos percentuais) e razão de riscos
d = 100 * (p - q)
print("Diferença de riscos:", round(d, 1), "pontos percentuais")
print("Razão de riscos:", round(p / q, 2))

# 4. Erro padrão e intervalo de 95% da diferença (aproximação normal)
z = NormalDist().inv_cdf(0.975)
se = 100 * sqrt(p * (1 - p) / n + q * (1 - q) / m)
itt_lower = d - z * se
itt_upper = d + z * se
print("Erro padrão:", round(se, 4), "; multiplicador normal:", round(z, 4))
print("Diferença:", round(d, 1), "; IC de 95%", round(itt_lower, 1), "a", round(itt_upper, 1))

# 5. A razão é tratada na escala do logaritmo e depois volta com exp
se_log = sqrt(1 / a - 1 / n + 1 / b - 1 / m)
print("Razão:", round(p / q, 2), "; IC de 95%", round(exp(log(p / q) - z * se_log), 2), "a", round(exp(log(p / q) + z * se_log), 2))

Pílula única: 199 de 857
Comprimidos separados: 279 de 859
Diferença de riscos: -9.3 pontos percentuais
Razão de riscos: 0.71
Erro padrão: 2.1525 ; multiplicador normal: 1.96
Diferença: -9.3 ; IC de 95% -13.5 a -5.0
Razão: 0.71 ; IC de 95% 0.61 a 0.84


### 10.4.3 Perguntas por protocolo e tratamento recebido

In [5]:
# 1. Manter só os adultos cujo tratamento recebido coincide com o braço alocado
matches = (trial["allocated"] == "single pill") == (trial["received"] == "yes")
kept = trial[matches]

# 2. Neste grupo restrito, contar eventos e adultos em cada braço
single = kept[kept["allocated"] == "single pill"]
separate = kept[kept["allocated"] == "separate tablets"]
a, n = single["y"].sum(), len(single)
b, m = separate["y"].sum(), len(separate)
print("Adultos mantidos:", len(kept), "; eventos/total:", a, "/", n, ",", b, "/", m)

# 3. Razão de riscos restrita e seu intervalo de 95% (escala do logaritmo)
ratio = (a / n) / (b / m)
se_log = sqrt(1 / a - 1 / n + 1 / b - 1 / m)
print("Razão restrita:", round(ratio, 2), "; IC de 95%", round(exp(log(ratio) - z * se_log), 2), "a", round(exp(log(ratio) + z * se_log), 2))

Adultos mantidos: 1610 ; eventos/total: 169 / 782 , 270 / 828
Razão restrita: 0.66 ; IC de 95% 0.56 a 0.78


### 10.5.3 Superioridade, não inferioridade e equivalência

*Que outros delineamentos poderiam responder à pergunta do diretor?*

In [6]:
# 1. Usar o intervalo da análise por intenção de tratar (seção 10.4.2)
print("Limite superior do intervalo de 95%:", round(itt_upper, 1), "pontos percentuais")

# 2. Superioridade: o intervalo inteiro fica abaixo de zero
print("Superioridade (limite superior abaixo de zero):", itt_upper < 0)

# 3. Não inferioridade: o limite superior fica abaixo da margem (margens ilustrativas)
for margin in [2, 5, 10]:
    print("Não inferioridade com margem", margin, "pontos:", itt_upper < margin)

# 4. Equivalência: o intervalo fica dentro da margem nos dois lados (margens ilustrativas)
for margin in [5, 10, 15]:
    print("Equivalência com margem", margin, "pontos:", (itt_lower > -margin) and (itt_upper < margin))

Limite superior do intervalo de 95%: -5.0 pontos percentuais
Superioridade (limite superior abaixo de zero): True
Não inferioridade com margem 2 pontos: True
Não inferioridade com margem 5 pontos: True
Não inferioridade com margem 10 pontos: True
Equivalência com margem 5 pontos: False
Equivalência com margem 10 pontos: False
Equivalência com margem 15 pontos: True


### 10.6.1 Ensaios randomizados por conglomerados

*Os próprios ambulatórios podem ser alocados ao acaso, até em etapas?*

In [7]:
# 1. Meses 13-18: as linhas dos ambulatórios que adotaram ao acaso no mês 13 ou ficaram com comprimidos separados
rollout = df[df["month"].between(13, 18)]

# 2. Agrupando por paciente: eventos e adultos em cada braço
for arm in ["single pill", "separate tablets"]:
    group = rollout[rollout["allocated"] == arm]
    print(arm + ":", group["clinic"].nunique(), "ambulatórios;", group["y"].sum(), "de", len(group))
single = rollout[rollout["allocated"] == "single pill"]
separate = rollout[rollout["allocated"] == "separate tablets"]
print("Razão de riscos agrupada:", round((single["y"].sum() / len(single)) / (separate["y"].sum() / len(separate)), 2))

# 3. Risco de cada ambulatório e depois a média dos riscos dos ambulatórios em cada braço
clinic_risk = rollout.groupby(["allocated", "clinic"])["y"].mean()
p_clinic = clinic_risk.loc["single pill"]
q_clinic = clinic_risk.loc["separate tablets"]
print("Médias dos riscos dos ambulatórios:", round(p_clinic.mean(), 4), ",", round(q_clinic.mean(), 4))

# 4. Intervalo de Welch para a diferença das médias dos ambulatórios (cada ambulatório é uma observação)
u = p_clinic.var() / len(p_clinic)
v = q_clinic.var() / len(q_clinic)
dof = (u + v) ** 2 / (u * u / (len(p_clinic) - 1) + v * v / (len(q_clinic) - 1))
d_clinic = 100 * (p_clinic.mean() - q_clinic.mean())
se_clinic = 100 * sqrt(u + v)
cut = t.ppf(0.975, dof)
print("Erro padrão:", round(se_clinic, 4), "; graus de liberdade:", round(dof, 2), "; multiplicador t:", round(cut, 4))
print("Diferença:", round(d_clinic, 1), "; IC de 95%", round(d_clinic - cut * se_clinic, 1), "a", round(d_clinic + cut * se_clinic, 1))

single pill: 5 ambulatórios; 179 de 819
separate tablets: 5 ambulatórios; 236 de 852
Razão de riscos agrupada: 0.79
Médias dos riscos dos ambulatórios: 0.2176 , 0.2776
Erro padrão: 1.6944 ; graus de liberdade: 8.0 ; multiplicador t: 2.3061
Diferença: -6.0 ; IC de 95% -9.9 a -2.1


### 10.6.2 Ensaios por conglomerados em degraus escalonados

In [8]:
# 1. Risco em cada grupo de ambulatórios e período (os que trocaram no mês 13 e no mês 19)
early = df[df["clinic_start_month"] == 13]
late = df[df["clinic_start_month"] == 19]
early_first = early[early["month"].between(13, 18)]["y"].mean()
early_second = early[early["month"].between(19, 24)]["y"].mean()
late_first = late[late["month"].between(13, 18)]["y"].mean()
late_second = late[late["month"].between(19, 24)]["y"].mean()
print("Trocaram no mês 13:", round(100 * early_first, 1), "%,", round(100 * early_second, 1), "%")
print("Trocaram no mês 19:", round(100 * late_first, 1), "%,", round(100 * late_second, 1), "%")

# 2. Mudança em cada grupo dos meses 13-18 para os meses 19-24
change_early = early_second - early_first
change_late = late_second - late_first
print("Mudanças:", round(100 * change_late, 1), ",", round(100 * change_early, 1), "pontos")

# 3. Contraste em degraus escalonados: a mudança dos ambulatórios tardios menos a dos precoces
print("Contraste em degraus escalonados:", round(100 * (change_late - change_early), 1), "pontos")

Trocaram no mês 13: 21.9 %, 19.5 %
Trocaram no mês 19: 27.7 %, 21.0 %
Mudanças: -6.7 , -2.4 pontos
Contraste em degraus escalonados: -4.3 pontos


### 10.7.1 Comparações antes e depois e tendências seculares

*O que os registros mostram sem alocação aleatória?*

In [9]:
# 1. Os ambulatórios que adotaram no mês 13: meses 1-6 (antes) e meses 13-18 (depois)
adopters = df[df["clinic_start_month"] == 13]
before = adopters[adopters["month"].between(1, 6)]
after = adopters[adopters["month"].between(13, 18)]

# 2. Somar as contagens antes de dividir
risk_before = before["y"].sum() / len(before)
risk_after = after["y"].sum() / len(after)
print("Antes:", before["y"].sum(), "/", len(before), "=", round(100 * risk_before, 1), "%")
print("Depois:", after["y"].sum(), "/", len(after), "=", round(100 * risk_after, 1), "%")
print("Razão depois/antes:", round(risk_after / risk_before, 2))

Antes: 311 / 830 = 37.5 %
Depois: 179 / 819 = 21.9 %
Razão depois/antes: 0.58


### 10.7.2 Comparações controladas e diferença em diferenças

In [10]:
# 1. Os quatro riscos: ambulatórios que adotaram (mês 13) e de comparação (mês 19), antes e depois
comparison = df[df["clinic_start_month"] == 19]
a0 = adopters[adopters["month"].between(1, 6)]["y"].mean()
a1 = adopters[adopters["month"].between(13, 18)]["y"].mean()
b0 = comparison[comparison["month"].between(1, 6)]["y"].mean()
b1 = comparison[comparison["month"].between(13, 18)]["y"].mean()
print("Ambulatórios que adotaram:", round(100 * a0, 1), "%,", round(100 * a1, 1), "%")
print("Ambulatórios de comparação:", round(100 * b0, 1), "%,", round(100 * b1, 1), "%")

# 2. Diferença em diferenças = mudança dos que adotaram menos mudança dos de comparação
print("Mudanças:", round(100 * (a1 - a0), 1), ",", round(100 * (b1 - b0), 1), "pontos")
print("Diferença em diferenças:", round(100 * ((a1 - a0) - (b1 - b0)), 1), "pontos")

# 3. O caminho suposto sem adoção e a razão de razões
print("Ponto final suposto sem adoção:", round(100 * (a0 + b1 - b0), 1), "%")
print("Razão de razões:", round((a1 / a0) / (b1 / b0), 2))

Ambulatórios que adotaram: 37.5 %, 21.9 %
Ambulatórios de comparação: 35.7 %, 27.7 %
Mudanças: -15.6 , -8.0 pontos
Diferença em diferenças: -7.6 pontos
Ponto final suposto sem adoção: 29.5 %
Razão de razões: 0.75


### 10.7.3 Série temporal interrompida

In [11]:
# 1. Contagens mensais dos ambulatórios que adotaram no mês 13
monthly = adopters.groupby("month")["y"].agg(["sum", "size"])

# 2. Número de meses antes, durante o ensaio e depois da adoção
print("Meses antes do ensaio:", (monthly.index < 7).sum())
print("Meses do ensaio:", ((monthly.index >= 7) & (monthly.index <= 12)).sum())
print("Meses depois da adoção:", (monthly.index >= 13).sum())

# 3. O primeiro risco mensal, com seu próprio denominador
first = monthly.iloc[0]
print("Primeiro mês:", first["sum"], "/", first["size"], "=", round(100 * first["sum"] / first["size"], 1), "%")

Meses antes do ensaio: 6
Meses do ensaio: 6
Meses depois da adoção: 12
Primeiro mês: 54 / 134 = 40.3 %


## Parte B. Soluções dos exercícios

### 10.10 Exercício 1. Equilíbrio basal

In [12]:
# 1. Pegar cada braço alocado e contar seus adultos
for arm in ["single pill", "separate tablets"]:
    group = trial[trial["allocated"] == arm]
    n = len(group)

    # 2. Contar os adultos com pressão sistólica basal de pelo menos 160 mmHg
    high = (group["baseline_sbp"] >= 160).sum()

    # 3. Contar os adultos com diabetes
    diabetes = (group["diabetes"] == "yes").sum()
    print(arm + ":", n, "adultos;", high, "com pressão basal de pelo menos 160 mmHg =", round(100 * high / n, 1), "%;",
          diabetes, "com diabetes =", round(100 * diabetes / n, 1), "%")

single pill: 857 adultos; 341 com pressão basal de pelo menos 160 mmHg = 39.8 %; 223 com diabetes = 26.0 %
separate tablets: 859 adultos; 331 com pressão basal de pelo menos 160 mmHg = 38.5 %; 207 com diabetes = 24.1 %


### 10.10 Exercício 3. Perguntas sobre alocação e adesão

In [13]:
# 1. The risks, the difference, and the ratio with their intervals (as in section 10.4.2)
single = trial[trial["allocated"] == "single pill"]
separate = trial[trial["allocated"] == "separate tablets"]
a, n = single["y"].sum(), len(single)
b, m = separate["y"].sum(), len(separate)
p = a / n
q = b / m
d = 100 * (p - q)
se = 100 * sqrt(p * (1 - p) / n + q * (1 - q) / m)
se_log = sqrt(1 / a - 1 / n + 1 / b - 1 / m)
print("Risks:", round(p, 4), ",", round(q, 4))
print("Diferença:", round(d, 1), "; IC de 95%", round(d - z * se, 1), "a", round(d + z * se, 1))
print("Razão:", round(p / q, 2), "; IC de 95%", round(exp(log(p / q) - z * se_log), 2), "a", round(exp(log(p / q) + z * se_log), 2))

# 2. How many adults received what they were assigned
matches = (trial["allocated"] == "single pill") == (trial["received"] == "yes")
print("Receipt matches assignment in", matches.sum(), "de", len(trial), "adults")

Risks: 0.2322 , 0.3248
Diferença: -9.3 ; IC de 95% -13.5 a -5.0
Razão: 0.71 ; IC de 95% 0.61 a 0.84
Receipt matches assignment in 1610 de 1716 adults


### 10.10 Exercício 6. Ambulatórios dentro dos grupos alocados

In [14]:
# 1. Each clinic in months 13-18: uncontrolled adults, adults, and risk
table = rollout.groupby(["allocated", "clinic"])["y"].agg(["sum", "size"])
table.columns = ["uncontrolled", "patients"]
table["risk (%)"] = round(100 * table["uncontrolled"] / table["patients"], 1)
display(table)

# 2. The Welch interval of the equal-clinic difference (as in section 10.6.1)
print("Diferença:", round(d_clinic, 1), "; IC de 95%", round(d_clinic - cut * se_clinic, 1), "a", round(d_clinic + cut * se_clinic, 1))

uncontrolled  patients  risk (%)
allocated        clinic                                  
separate tablets C01               50       167      29.9
                 C02               42       180      23.3
                 C05               50       167      29.9
                 C07               45       163      27.6
                 C10               49       175      28.0
single pill      C03               37       167      22.2
                 C04               35       171      20.5
                 C06               36       160      22.5
                 C08               27       148      18.2
                 C09               44       173      25.4

Diferença: -6.0 ; IC de 95% -9.9 a -2.1


### 10.10 Exercício 7. A mudança no serviço inteiro

In [15]:
# 1. Meses 1-6 e meses 19-24 para o serviço inteiro
periods = [("months 1-6", df[df["month"].between(1, 6)]), ("months 19-24", df[df["month"].between(19, 24)])]

# 2. Em cada período: adultos, não controlados e os que receberam a pílula única
for name, rows_in_period in periods:
    n = len(rows_in_period)
    uncontrolled = rows_in_period["y"].sum()
    uptake = (rows_in_period["received"] == "yes").sum()
    print(name + ": não controlados", uncontrolled, "/", n, "=", round(100 * uncontrolled / n, 1), "%; uso da pílula única", uptake, "/", n, "=", round(100 * uptake / n, 1), "%")

months 1-6: não controlados 621 / 1698 = 36.6 %; uso da pílula única 56 / 1698 = 3.3 %
months 19-24: não controlados 336 / 1659 = 20.3 %; uso da pílula única 1527 / 1659 = 92.0 %


### 10.10 Exercício 8. O contraste em degraus escalonados contra a comparação por conglomerados

In [16]:
# 1. O contraste em degraus escalonados (seção 10.6.2) e a comparação por conglomerados dos meses 13-18 agrupada por paciente
print("Contraste em degraus escalonados:", round(100 * (change_late - change_early), 1), "pontos")
print("Comparação por conglomerados (agrupada por paciente):", round(100 * (early_first - late_first), 1), "pontos")

Contraste em degraus escalonados: -4.3 pontos
Comparação por conglomerados (agrupada por paciente): -5.8 pontos


---
*Dados sintéticos (semente 33, `generate_data.py`). Valores apenas didáticos; nada aqui é recomendação clínica nem estimativa para um serviço real.*